In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pandas import DataFrame
from subprocess import call
from matplotlib.pyplot import imread, imshow, axis, figure, show

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, make_scorer
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

import shap
from matplotlib import pyplot as plt
from matplotlib.transforms import Bbox

In [ ]:
dataset = pd.read_excel("895_records_with_descriptors_with_names.xlsx")
dataset.shape

##### **Variables Enconding**

In [ ]:
dissolve_encoding: dict[str, int] = {"NO": 0,"YES": 1}
sample_type_enconding: dict[str, int] = {"pellet": 0, "waste": 1, "fiber": 2, "film": 3, "powder": 4}

encoding: dict[str, dict[str, int]] = {
    "Dissolution": dissolve_encoding,
    "Sample type": sample_type_enconding
}
df: DataFrame = dataset.replace(encoding, inplace=False)

##### **Variables Normalization**

In [4]:
# Remove Polymer and Solvent Identifier Columns
df = df.drop(columns=["Polymer_ID", "Solvent_ID", "Polymer", "Solvent"])

# Separate features (X) from target "Dissolution" (y)
X = df.drop(columns=["Dissolution"])
y = df["Dissolution"]

# Apply log(1+x) Transformation
X_log = np.log1p(X)

# Scale to [0,1] with MinMax Normalization
min_max_scaler = MinMaxScaler(feature_range=(0, 1), copy=True)
X_scaled = min_max_scaler.fit_transform(X_log)

# Rebuild Dataset merging normalized features with target
df_log_minmax = DataFrame(X_scaled, columns = X.columns, index= X.index)
df_log_minmax["Dissolution"] = y

In [5]:
# Separate features (X) from target "Dissolve" (y)
X = df_log_minmax.drop(columns=["Dissolution"])
y = df_log_minmax["Dissolution"]

##### **Evaluation Decision Tree Model**

With a defined random seed and for the best results of 5-Fold CV:

In [ ]:
# Hyperparameters to test
param_grid = {
    'criterion': 'entropy',
    'max_depth': 6,
    'max_features': None,
    'max_leaf_nodes': None,
    'min_impurity_decrease': 0.0,
    'min_samples_leaf': 1,
    'min_samples_split': 4,
    'splitter': 'best',
}

# Save Results
acc_val_list = []
acc_train_list = []
acc_test_list = []
recall_test_list = []
best_params_list = []

# Loop of Iterations with random_state varying the data split 
for i in range(5): 
    # Split data into Train (70%)
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=i, stratify=y)
    # Split the rest of the data into Validation (15%) and Test (15%)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=i, stratify=y_temp)

    print(f"\n{'='*30}\nStarting Iteration {i+1}\n{'='*30}")

    # Train model with hyperparameters
    model = DecisionTreeClassifier(**param_grid, random_state=4)
    model.fit(X_train, y_train)
    print(model.max_features_)

    # Predictions
    y_val_pred = model.predict(X_val)
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Compute Metrics
    acc_val = accuracy_score(y_val, y_val_pred)
    acc_train = accuracy_score(y_train, y_train_pred)
    acc_test = accuracy_score(y_test, y_test_pred)

    # Store metrics
    acc_val_list.append(acc_val)
    acc_train_list.append(acc_train)
    acc_test_list.append(acc_test)

    # Show iteration results
    print(f"Running parameters:  {param_grid}")
    print(f"Accuracy - Validation:  {acc_val:.4f}")
    print(f"Accuracy - Train:     {acc_train:.4f}")
    print(f"Accuracy - Test:      {acc_test:.4f}")


# Compute mean and standard deviation after all iterations
print("\n=== Final Results ===")
print(f"Accuracy - Validation: Mean = {np.mean(acc_val_list):.4f}, Standard Deviation = {np.std(acc_val_list):.4f}")
print(f"Accuracy - Train:     Mean = {np.mean(acc_train_list):.4f}, Standard Deviation = {np.std(acc_train_list):.4f}")
print(f"Accuracy - Test:      Mean = {np.mean(acc_test_list):.4f}, Standard Deviation = {np.std(acc_test_list):.4f}")

DT model tree structure

In [ ]:

feature_names_input = X.columns.tolist()
st_labels_input = [str(label) for label in sorted(y.unique())]

# --- Parameters ---
param_grid = {'criterion': 'entropy', 'max_depth': 6, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 4, 'splitter': 'best'}

file_tag = "rs3_full"
eval_metric = "accuracy"
tree_filename = f"./images/{file_tag}_DT"
max_depth2show = 6

os.makedirs("images", exist_ok=True)


best_model = DecisionTreeClassifier(**param_grid, random_state=4)
best_model.fit(X, y)


export_graphviz(
    best_model,
    out_file=tree_filename + ".dot",
    max_depth=max_depth2show,
    feature_names=feature_names_input,
    class_names=st_labels_input,
    filled=True,
    rounded=True,
    impurity=False,
    special_characters=True,
    precision=2,
)

call([
    "dot", "-Tpng",
    tree_filename + ".dot",
    "-o", tree_filename + ".png",
    "-Gdpi=400"
])

figure(figsize=(14, 6))
imshow(imread(tree_filename + ".png"))
axis("off")
show()

XAI

Relative importance of the 10 most important variables

In [ ]:

feature_names = X.columns.tolist()

# 1. Get importances and convert to percentages
importances = best_model.feature_importances_
indices = np.argsort(importances)[::-1]  

# 2. Select only the top 10
top_n = 10
indices = indices[:top_n]
elems = [feature_names[i] for i in indices]
imp_values = [importances[i] * 100 for i in indices]  # convert to %
green_color = "#77AD6F"

# 3. Create figure
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(elems, imp_values, color=green_color, edgecolor='black', linewidth=0.5)
for bar, value in zip(bars, imp_values):
    ax.text(
        bar.get_width() + 0.5,  # slightly to the right of the bar
        bar.get_y() + bar.get_height() / 2,
        f"{value:.1f}%",
        va='center',
        ha='left',
        fontsize=9,
        color='black'
    )
max_value = max(imp_values)
ax.set_xlim(0, max_value * 1.15)  
ax.tick_params(axis='x', width=0.5, length=4)
ax.tick_params(axis='y', width=0.5, length=4)
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.8)

ax.grid(False)

# 4. Set title and labels
ax.set_title("", fontsize=14, weight='bold')
ax.set_xlabel("Importance (%)", fontsize=12)
ax.set_ylabel("Variables", fontsize=12)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

output_path = f"images/_dt_top10_vars_ranking.png"
fig.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"Chart saved to: {output_path}")


##### **Gradient Boosting Model**

With a defined random seed and for the best results of 5-Fold CV:

In [ ]:
# Hyperparameters to test
param_grid = {
    'learning_rate': 0.1,
    'loss': 'log_loss',
    'max_depth': 4,
    'max_features': 'sqrt',
    'min_samples_leaf': 1,
    'min_samples_split': 2,
    'n_estimators': 15,
    'subsample': 0.9
}

# Save Results
acc_val_list = []
acc_train_list = []
acc_test_list = []
recall_test_list = []
best_params_list = []

# Loop of Iterations with random_state varying the data split 
for i in range(5): 
    # Split data into Train (70%)
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=i, stratify=y)
    # Split the rest of the data into Validation (15%) and Test (15%)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=i, stratify=y_temp)

    print(f"\n{'='*30}\nStarting Iteration {i+1}\n{'='*30}")

    # Train model with hyperparameters
    gb_model = GradientBoostingClassifier(**param_grid, random_state=4)
    gb_model.fit(X_train, y_train)

    # Predictions
    y_val_pred = gb_model.predict(X_val)
    y_train_pred = gb_model.predict(X_train)
    y_test_pred = gb_model.predict(X_test)

    # Compute Metrics
    acc_val = accuracy_score(y_val, y_val_pred)
    acc_train = accuracy_score(y_train, y_train_pred)
    acc_test = accuracy_score(y_test, y_test_pred)

    # Store metrics
    acc_val_list.append(acc_val)
    acc_train_list.append(acc_train)
    acc_test_list.append(acc_test)

    # Show iteration results
    print(f"Running parameters:  {param_grid}")
    print(f"Accuracy - Validation:  {acc_val:.4f}")
    print(f"Accuracy - Train:     {acc_train:.4f}")
    print(f"Accuracy - Test:      {acc_test:.4f}")


# Compute mean and standard deviation after all iterations
print("\n=== Final Results ===")
print(f"Accuracy - Validation: Mean = {np.mean(acc_val_list):.4f}, Standard Deviation = {np.std(acc_val_list):.4f}")
print(f"Accuracy - Train:     Mean = {np.mean(acc_train_list):.4f}, Standard Deviation = {np.std(acc_train_list):.4f}")
print(f"Accuracy - Test:      Mean = {np.mean(acc_test_list):.4f}, Standard Deviation = {np.std(acc_test_list):.4f}")

XAI

Relative importance of the 10 most important variables

In [ ]:
feature_names = X.columns.tolist()

param_grid = {
    'learning_rate': 0.1,
    'loss': 'log_loss',
    'max_depth': 4,
    'max_features': 'sqrt',
    'min_samples_leaf': 1,
    'min_samples_split': 2,
    'n_estimators': 15,
    'subsample': 0.9,
    'random_state': 4
}

gb_model = GradientBoostingClassifier(**param_grid)
gb_model.fit(X, y)

# 1. Get importances and convert to percentages

importances = gb_model.feature_importances_
indices = np.argsort(importances)[::-1]  

# 2. Select only the top 10 variables
top_n = 10
indices = indices[:top_n]
elems = [feature_names[i] for i in indices]
imp_values = [importances[i] * 100 for i in indices]  # convert to %
green_color = "#77AD6F"

# 3. Create figure
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(elems, imp_values, color=green_color, edgecolor='black', linewidth=0.5)
for bar, value in zip(bars, imp_values):
    ax.text(
        bar.get_width() + 0.5,  
        bar.get_y() + bar.get_height() / 2,
        f"{value:.1f}%",
        va='center',
        ha='left',
        fontsize=12,
        color='black'
    )
max_value = max(imp_values)
ax.set_xlim(0, max_value * 1.15)
ax.tick_params(axis='x', width=0.5, length=4)
ax.tick_params(axis='y', width=0.5, length=4)
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.8)

ax.grid(False)

# 4. Set title and labels
ax.set_title("", fontsize=14, weight='bold')
ax.set_xlabel("Importance (%)", fontsize=12)
ax.set_ylabel("Variables", fontsize=12)


plt.tight_layout()
plt.show()


output_path = f"images/_gb_top10_vars_ranking.png"
fig.savefig(output_path, dpi=300, bbox_inches='tight')



SHAP

In [11]:
explainer = shap.Explainer(gb_model, X)
shap_values = explainer(X) 

In [ ]:
# Beeswarm plot
plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_values, max_display=20)
plt.tight_layout()

plt.savefig("beeswarm_plot.png", dpi=300, bbox_inches='tight')

plt.show()


MLP

With a defined random seed and for the best results of 5-Fold CV:

In [ ]:

# === Hiperparâmetros da MLP ===
mlp_params = {
    'hidden_layer_sizes': (50,),
    'activation': 'relu',
    'alpha': 0.001,               
    'learning_rate': 'constant',
    'learning_rate_init': 0.5,
    'solver': 'sgd',
    'max_iter': 200
}

# === Listas para armazenar resultados ===
acc_val_list = []
acc_train_list = []
acc_test_list = []
recall_val_list = []
recall_train_list = []
recall_test_list = []

# === Loop de 5 iterações com splits diferentes ===
for i in range(5):
    # Split data em treino (70%), validação (15%) e teste (15%)
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=i, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=i, stratify=y_temp)

    print(f"\n{'='*30}\nStarting Iteration {i+1}\n{'='*30}")

    # === Treina a MLP ===
    mlp_model = MLPClassifier(**mlp_params, random_state=2)
    mlp_model.fit(X_train, y_train)

    # === Previsões ===
    y_val_pred = mlp_model.predict(X_val)
    y_train_pred = mlp_model.predict(X_train)
    y_test_pred = mlp_model.predict(X_test)

    # === Métricas ===
    acc_val = accuracy_score(y_val, y_val_pred)
    acc_train = accuracy_score(y_train, y_train_pred)
    acc_test = accuracy_score(y_test, y_test_pred)

    # Usa recall binário (ou 'macro' se for multiclasse)
    recall_val = recall_score(y_val, y_val_pred, average='binary')
    recall_train = recall_score(y_train, y_train_pred, average='binary')
    recall_test = recall_score(y_test, y_test_pred, average='binary')

    # === Guarda resultados ===
    acc_val_list.append(acc_val)
    acc_train_list.append(acc_train)
    acc_test_list.append(acc_test)

    recall_val_list.append(recall_val)
    recall_train_list.append(recall_train)
    recall_test_list.append(recall_test)

    # === Mostra resultados da iteração ===
    print(f"Running parameters:  {mlp_params}")
    print(f"Accuracy - Validation:  {acc_val:.4f}")
    print(f"Accuracy - Train:       {acc_train:.4f}")
    print(f"Accuracy - Test:        {acc_test:.4f}")
    print(f"Recall - Validation:    {recall_val:.4f}")
    print(f"Recall - Train:         {recall_train:.4f}")
    print(f"Recall - Test:          {recall_test:.4f}")

# === Resultados finais ===
print("\n=== Final Results (Mean ± Std) ===")
print(f"Accuracy - Validation: Mean = {np.mean(acc_val_list):.4f}, Std = {np.std(acc_val_list):.4f}")
print(f"Accuracy - Train:     Mean = {np.mean(acc_train_list):.4f}, Std = {np.std(acc_train_list):.4f}")
print(f"Accuracy - Test:      Mean = {np.mean(acc_test_list):.4f}, Std = {np.std(acc_test_list):.4f}")

print(f"Recall - Validation:  Mean = {np.mean(recall_val_list):.4f}, Std = {np.std(recall_val_list):.4f}")
print(f"Recall - Train:       Mean = {np.mean(recall_train_list):.4f}, Std = {np.std(recall_train_list):.4f}")
print(f"Recall - Test:        Mean = {np.mean(recall_test_list):.4f}, Std = {np.std(recall_test_list):.4f}")


XAI

SHAP analysis

In [ ]:
mlp_params = {
    'hidden_layer_sizes': (50,),
    'activation': 'relu',
    'alpha': 0.001,               
    'learning_rate': 'constant',
    'learning_rate_init': 0.5,
    'solver': 'sgd',
    'max_iter': 200
}

mlp_model = MLPClassifier(**mlp_params, random_state=2)
mlp_model.fit(X, y)

background = shap.sample(X, 600, random_state=3)
explainer = shap.KernelExplainer(mlp_model.predict_proba, background)
explanation = explainer(X)   
shap_values_class0 = explanation[:, :, 0]
shap_values_class1 = explanation[:, :, 1]

In [ ]:

def set_all_text_black(fig):
    
    for ax in fig.axes:
        ax.title.set_color("black")
        ax.xaxis.label.set_color("black")
        ax.yaxis.label.set_color("black")
        for lab in ax.get_xticklabels() + ax.get_yticklabels():
            lab.set_color("black")

def add_bottom_line(fig, lw=0.8, pad=0.004):
    
    main_axes = fig.axes[:-1] if len(fig.axes) > 1 else fig.axes
    boxes = [ax.get_position() for ax in main_axes]
    bbox = boxes[0]
    for b in boxes[1:]:
        bbox = Bbox.union([bbox, b])


    x0 = max(0, bbox.x0 - pad)
    x1 = min(1, bbox.x1 + pad)
    y0 = max(0, bbox.y0 - pad)

    
    fig.add_artist(plt.Line2D(
        [x0, x1], [y0, y0],
        transform=fig.transFigure,
        color="black", lw=lw, zorder=1000
    ))

# ===========================================
# 1. Beeswarm plot 
# ===========================================

fig = plt.figure(figsize=(8, 5))
plt.subplots_adjust(right=0.80)  

shap.plots.beeswarm(shap_values_class1, max_display=20, show=False)

set_all_text_black(fig)


plt.savefig("shap_beeswarm_bottomline.png", dpi=300, bbox_inches="tight")
plt.show()


fig = plt.figure(figsize=(8, 5))
shap.plots.bar(shap_values_class1, max_display=20, show=False)

set_all_text_black(fig)

plt.savefig("shap_bar_bottomline.png", dpi=300, bbox_inches="tight")
plt.show()